# Purchase-intent data exploration

A reproducible first look at conversion imbalance, session behavior, seasonality, and visitor cohorts. `PageValues` is retained for late-session prediction and should not be presented as an early-session signal.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

from src.config import ProjectConfig
from src.data import load_data

sns.set_theme(style="whitegrid", palette="crest")
config = ProjectConfig.load(ROOT / "config.yaml")
sessions = load_data(config=config)
sessions.head()

## Conversion balance

Purchases are the minority class, so accuracy alone is not a useful success metric. The modeling pipeline foregrounds PR-AUC and buyer recall.

In [ ]:
conversion_rate = sessions["Converted"].mean()
print(f"Sessions: {len(sessions):,} | purchase rate: {conversion_rate:.1%}")
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=sessions, x="Converted", hue="Converted", legend=False, ax=ax)
ax.set(title="Purchases are the minority outcome", xlabel="Converted (0 = no, 1 = yes)", ylabel="Sessions")
plt.show()

**Takeaway:** the positive class is materially smaller; compare candidate models with PR-AUC, recall, and a business-selected threshold.

In [ ]:
behavior = sessions.groupby("Converted", observed=True).agg(
    sessions=("Converted", "size"),
    median_page_value=("PageValues", "median"),
    median_product_pages=("ProductRelated", "median"),
    median_exit_rate=("ExitRates", "median"),
    median_product_time=("ProductRelated_Duration", "median"),
)
behavior

**Takeaway:** stronger product engagement and page value can separate purchases, while `PageValues`' late availability makes this a late-session model.

In [ ]:
month_order = config.data["months"]
monthly = sessions.groupby("Month", observed=True)["Converted"].agg(
    sessions="size", conversion_rate="mean", purchases="sum"
).reindex(month_order).dropna().reset_index()
fig = px.line(
    monthly, x="Month", y="conversion_rate", markers=True,
    hover_data=["sessions", "purchases"],
    title="Conversion rate varies with calendar context",
    labels={"conversion_rate": "Conversion rate"},
)
fig.update_yaxes(tickformat=".0%")
fig.show()

**Takeaway:** use month and special-day context as categorical/calendar signals; validate seasonal performance before deployment.

In [ ]:
special_day = sessions.groupby("SpecialDay", observed=True)["Converted"].agg(
    sessions="size", conversion_rate="mean"
).reset_index()
fig = px.bar(
    special_day, x="SpecialDay", y="conversion_rate", color="conversion_rate",
    hover_data=["sessions"], title="Conversion by special-day proximity",
    labels={"conversion_rate": "Conversion rate", "SpecialDay": "Proximity"},
)
fig.update_yaxes(tickformat=".0%")
fig.show()

**Takeaway:** special-day proximity has a relationship with purchases that warrants validation across calendar periods and campaigns.

In [ ]:
cohorts = sessions.groupby("VisitorType", observed=True)["Converted"].agg(
    sessions="size", conversion_rate="mean"
).sort_values("conversion_rate", ascending=False)
cohorts

**Takeaway:** returning and new visitors can have different baseline intent; assess cohort performance rather than applying one story to all visits.

In [ ]:
numeric = sessions.select_dtypes(include="number").drop(columns="Converted")
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(numeric.corr(), cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation structure of numeric session signals")
plt.show()

sample = sessions.sample(min(1800, len(sessions)), random_state=config.project["random_seed"])
interactive = px.scatter(
    sample, x="ExitRates", y="PageValues", color="Converted",
    opacity=0.5, title="Exit behavior and page values by outcome",
    hover_data=["ProductRelated", "VisitorType", "Month"],
)
interactive.show()

**Takeaway:** correlated browsing metrics should be interpreted carefully; inspect feature importance and local explanations, and treat predictive associations as non-causal.